# Module 4: LLM Evaluation

**Goal:** Build a multi-layer evaluation system — automated metrics, LLM-as-judge, and RAG-specific scoring.

**What you'll learn:**
1. Why LLM evaluation is hard (and different from classical ML)
2. Reference-based metrics: ROUGE, BLEU, BERTScore
3. LLM-as-judge: scoring without a ground truth
4. RAG-specific metrics: faithfulness, context relevance, answer relevance
5. Building a reusable eval harness
6. A/B testing two prompts systematically

---

## 0. Setup

In [ ]:
%pip install -q openai evaluate rouge-score python-dotenv

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from openai import OpenAI
client = OpenAI()

def llm(prompt: str, system: str = None, model="gpt-4o-mini", temperature=0.0) -> str:
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    return client.chat.completions.create(model=model, messages=msgs, temperature=temperature).choices[0].message.content

print("✅ Ready")

---
## 1. Why Evaluation Is Hard for LLMs

In classical ML: loss goes down → model improves. Simple.

In LLMs:
- Two correct answers can be phrased completely differently (ROUGE would score one wrong)
- "Correct" often requires world knowledge, not just string matching
- The model being evaluated might have memorised the benchmark
- User satisfaction ≠ automated metric score

**The evaluation pyramid** (use all three layers):
```
         [User satisfaction]     ← hardest, most meaningful
        [LLM-as-judge scores]    ← scalable, surprisingly good
    [Automated metrics: ROUGE...]← cheap, fast, but limited
```

---
## 2. Reference-Based Metrics

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

# Ground truth vs. model output pairs
examples = [
    {
        "reference": "The Eiffel Tower is located in Paris, France.",
        "candidate_good": "The Eiffel Tower stands in Paris, the capital of France.",
        "candidate_bad": "The Eiffel Tower was built in 1889 by Gustave Eiffel.",  # True but not answering location
    },
    {
        "reference": "Python uses indentation to define code blocks.",
        "candidate_good": "In Python, code blocks are defined using indentation rather than braces.",
        "candidate_bad": "Python is a popular programming language used for data science.",
    },
]

for i, ex in enumerate(examples):
    good_score = rouge.compute(predictions=[ex["candidate_good"]], references=[ex["reference"]])
    bad_score  = rouge.compute(predictions=[ex["candidate_bad"]],  references=[ex["reference"]])
    print(f"Example {i+1}:")
    print(f"  Good answer ROUGE-L: {good_score['rougeL']:.3f}")
    print(f"  Bad answer  ROUGE-L: {bad_score['rougeL']:.3f}")
    print()

In [ ]:
# The fundamental ROUGE limitation: paraphrases score poorly
perfect_paraphrase = "Paris, France is where you'll find the Eiffel Tower."
reference = "The Eiffel Tower is located in Paris, France."

score = rouge.compute(predictions=[perfect_paraphrase], references=[reference])
print(f"Perfect paraphrase ROUGE-L: {score['rougeL']:.3f}")
print("⚠️  Low score despite being semantically identical — this is why ROUGE alone isn't enough")

---
## 3. LLM-as-Judge

Use a stronger model (GPT-4) to evaluate outputs from a weaker model. Scales infinitely, no reference needed, handles paraphrases naturally.

In [ ]:
def judge_response(
    question: str,
    answer: str,
    criteria: dict = None,
    reference: str = None,
) -> dict:
    """
    LLM-as-judge evaluation.
    Returns scores for accuracy, completeness, clarity (1-5 each).
    """
    criteria = criteria or {"accuracy": "Is it factually correct?", "completeness": "Does it fully answer?", "clarity": "Is it clear and readable?"}
    
    ref_block = f"\nReference answer: {reference}" if reference else ""
    criteria_text = "\n".join(f'  "{k}": <1-5>  // {v}' for k, v in criteria.items())

    prompt = f"""You are a strict evaluator. Score this answer on each criterion from 1 (terrible) to 5 (excellent).

Question: {question}
Answer: {answer}{ref_block}

Return JSON only:
{{
{criteria_text},
  "reasoning": "<1 sentence>"
}}"""

    raw = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    ).choices[0].message.content
    return json.loads(raw)


# Test it
question = "Explain what a neural network is."
good_answer = "A neural network is a system of interconnected nodes (neurons) arranged in layers. It learns by adjusting connection weights based on examples, enabling it to recognise patterns."
bad_answer = "Neural networks are very complex. They use AI to do machine learning stuff."

good_scores = judge_response(question, good_answer)
bad_scores  = judge_response(question, bad_answer)

print("Good answer scores:", {k: v for k, v in good_scores.items() if k != 'reasoning'})
print("Bad answer scores: ", {k: v for k, v in bad_scores.items() if k != 'reasoning'})
print(f"\nGood reasoning: {good_scores['reasoning']}")
print(f"Bad reasoning:  {bad_scores['reasoning']}")

---
## 4. A/B Testing Two Prompts

Systematically compare two prompt strategies across a test set.

In [ ]:
test_questions = [
    "What is gradient descent?",
    "Explain overfitting in machine learning.",
    "What is the difference between supervised and unsupervised learning?",
    "How does backpropagation work?",
    "What is a transformer model?",
]

PROMPT_A = "Answer this question briefly: {question}"
PROMPT_B = """You are an expert ML tutor. Answer this question for a software engineer new to ML.
Use one analogy, be concise (3-4 sentences max).

Question: {question}"""

results = []
for q in test_questions:
    answer_a = llm(PROMPT_A.format(question=q))
    answer_b = llm(PROMPT_B.format(question=q))
    
    score_a = judge_response(q, answer_a)
    score_b = judge_response(q, answer_b)
    
    avg_a = sum(v for k, v in score_a.items() if k != "reasoning") / (len(score_a) - 1)
    avg_b = sum(v for k, v in score_b.items() if k != "reasoning") / (len(score_b) - 1)
    
    results.append({"question": q[:40], "avg_a": round(avg_a, 2), "avg_b": round(avg_b, 2), "winner": "B" if avg_b > avg_a else "A"})
    print(f"Q: {q[:40]:40s}  A={avg_a:.1f}  B={avg_b:.1f}  → {'B wins' if avg_b > avg_a else 'A wins' if avg_a > avg_b else 'Tie'}")

wins_a = sum(1 for r in results if r["winner"] == "A")
wins_b = sum(1 for r in results if r["winner"] == "B")
print(f"\nFinal: Prompt A wins {wins_a}/5  |  Prompt B wins {wins_b}/5")

---
## 5. Building a Reusable Eval Harness

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, List
import statistics

@dataclass
class EvalCase:
    question: str
    reference: str = None
    tags: List[str] = field(default_factory=list)

@dataclass
class EvalResult:
    case: EvalCase
    answer: str
    scores: dict

class EvalHarness:
    """Run a test suite against any LLM function and report results."""

    def __init__(self, model_fn: Callable[[str], str], name: str = "model"):
        self.model_fn = model_fn
        self.name = name
        self.results: List[EvalResult] = []

    def run(self, cases: List[EvalCase]) -> "EvalHarness":
        self.results = []
        for case in cases:
            answer = self.model_fn(case.question)
            scores = judge_response(case.question, answer, reference=case.reference)
            self.results.append(EvalResult(case=case, answer=answer, scores=scores))
        return self

    def report(self):
        metrics = [k for k in self.results[0].scores if k != "reasoning"]
        print(f"\n{'='*60}")
        print(f"Eval report: {self.name} ({len(self.results)} cases)")
        print('='*60)
        for metric in metrics:
            vals = [r.scores[metric] for r in self.results]
            print(f"  {metric:20s}  avg={statistics.mean(vals):.2f}  min={min(vals)}  max={max(vals)}")
        return self


# Define test suite
eval_cases = [
    EvalCase("What is RAG?", reference="RAG combines retrieval with generation to ground LLM answers in external knowledge.", tags=["fundamentals"]),
    EvalCase("Name 3 vector databases.", tags=["infrastructure"]),
    EvalCase("What is hallucination in LLMs?", tags=["fundamentals"]),
]

# Run eval
harness = EvalHarness(
    model_fn=lambda q: llm(f"Answer concisely: {q}"),
    name="gpt-4o-mini baseline"
)
harness.run(eval_cases).report()

---
## 6. Eval Best Practices

| ✅ Do | ❌ Don't |
|-------|----------|
| Fix your eval set before changing the model | Tune on eval data (leakage) |
| Use multiple evaluators (metrics + judge) | Trust a single metric |
| Track scores over time in a database | Re-run evals from scratch each time |
| Evaluate on failure cases specifically | Only eval on easy examples |
| Blind evaluation (hide which is A/B) | Let the judge see which model it's rating |

## 🧪 Exercises

1. **Positional bias**: Run `judge_response` on the same answer twice but swap the order A/B. Does the judge prefer whichever comes first? This is a known LLM judge failure mode.
2. **Verbosity bias**: Generate a short (2-sentence) and long (10-sentence) version of the same answer. Does the judge rate the longer one higher regardless of quality?
3. **Add a custom criterion**: Add `"conciseness"` to the judge criteria. Re-run the A/B test — does the winner change?
4. **Build a golden dataset**: Create 10 question–reference pairs for a topic you know well. Run the harness and find your model's weakest category.

---
**Next:** [Module 05 — Deployment](../05-deployment/deployment.ipynb)